# ⚙️ LoRA (Low-Rank Adaptation) 밑바닥부터 구현하기

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import logging
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

import transformers
transformers.logging.set_verbosity_error()

import torch
import torch.nn as nn
import math

## 1. 일반적인 Linear Layer (원본 가중치)
먼저, 파라미터가 10,000 x 10,000개인 거대한 선형 레이어(Linear Layer)를 가정해 봅시다.

In [2]:
# 거대한 모델의 한 층이라고 가정
in_features = 10000
out_features = 10000

# 원본 가중치 행렬 레이어 생성
original_layer = nn.Linear(in_features, out_features, bias=False)

# 파라미터 개수 계산
full_params = sum(p.numel() for p in original_layer.parameters() if p.requires_grad)
print(f"Full Fine-Tuning 시 학습해야 할 파라미터 수: {full_params:,} 개")

Full Fine-Tuning 시 학습해야 할 파라미터 수: 100,000,000 개


## 2. LoRA Layer 구현 (A, B 행렬 쪼개기)
원본 가중치는 동결(Freeze)하고, 랭크 $r$을 가지는 두 개의 작은 행렬 A와 B를 병렬로 추가하는 LoRA 층을 구현합니다.

In [3]:
class LoRALayer(nn.Module):
    def __init__(self, original_layer, rank=8, alpha=16):
        super(LoRALayer, self).__init__()
        # 1. 원본 가중치 보존 및 동결 (학습 금지)
        self.original_layer = original_layer
        for param in self.original_layer.parameters():
            param.requires_grad = False
            
        self.rank = rank
        self.scaling = alpha / rank  # LoRA 논문에서 제안한 스케일링 팩터
        
        in_dim = original_layer.in_features
        out_dim = original_layer.out_features
        
        # 2. 행렬 A: (in_dim x rank) - Kaiming 초기화 (랜덤)
        self.lora_A = nn.Linear(in_dim, rank, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        
        # 3. 행렬 B: (rank x out_dim) - 0으로 초기화
        # 처음에는 B 행렬이 0이므로 원본 모델의 결과값에 아무런 영향을 주지 않습니다. (안전한 출발)
        self.lora_B = nn.Linear(rank, out_dim, bias=False)
        nn.init.zeros_(self.lora_B.weight)
        
    def forward(self, x):
        # h = Wx + (BAx) * scaling
        original_output = self.original_layer(x)
        lora_output = self.lora_B(self.lora_A(x)) * self.scaling
        
        return original_output + lora_output

# 원본 레이어를 LoRA 레이어로 래핑 (랭크 r=8 설정)
lora_wrapped_layer = LoRALayer(original_layer, rank=8)

# 파라미터 개수 비교 계산
lora_params = sum(p.numel() for p in lora_wrapped_layer.parameters() if p.requires_grad)

print(f"LoRA 적용 후 학습해야 할 파라미터 수: {lora_params:,} 개")
print(f"👉 파라미터가 전체 대비 {lora_params / full_params * 100:.3f}% 로 극적으로 감소했습니다!")

LoRA 적용 후 학습해야 할 파라미터 수: 160,000 개
👉 파라미터가 전체 대비 0.160% 로 극적으로 감소했습니다!


## 3. 순전파(Forward Pass) 연산 차원 검증
입력 데이터가 들어왔을 때, 기존 Wx 연산과 LoRA의 Wx + BAx 연산의 출력 차원이 동일하게 유지되는지 확인합니다.

In [4]:
# 배치 크기 32를 가정한 더미 입력 데이터
dummy_input = torch.randn(32, in_features)

# 원본 레이어 통과
out_original = original_layer(dummy_input)

# LoRA 레이어 통과
out_lora = lora_wrapped_layer(dummy_input)

print(f"입력 차원: {dummy_input.shape}")
print(f"원본 레이어 출력 차원: {out_original.shape}")
print(f"LoRA 레이어 출력 차원: {out_lora.shape}")

# B가 0으로 초기화되었기 때문에, 학습 전 첫 출력값은 원본과 완벽히 동일해야 합니다.
is_equal = torch.allclose(out_original, out_lora)
print(f"초기 출력값이 원본과 동일합니까? {is_equal}")

입력 차원: torch.Size([32, 10000])
원본 레이어 출력 차원: torch.Size([32, 10000])
LoRA 레이어 출력 차원: torch.Size([32, 10000])
초기 출력값이 원본과 동일합니까? True
